# Masked-Atom Pretraining + Fine-Tuning for MDM2 (GNN)

> **Your 3-step idea, implemented end-to-end on YOUR data (645 MDM2 compounds + 154k COCONUT SMILES).**
>
> 1. **The Masking Task** — hide one atom in a real molecule, make the GNN guess the element from neighbours only.
> 2. **The Result** — the model learns standard chemistry rules ("N is usually next to C here").
> 3. **Fine-Tuning** — load your ~600 compounds, train on the real target (pIC50 + Active/Inactive). Because the encoder already knows chemistry, it trains **faster and better** than from scratch.

**What this notebook proves (on your machine/Colab):**

| Stage | Data | Labels needed? | Output |
|---|---|---|---|
| A. Self-supervised pretraining | COCONUT SMILES (unlabeled, up to 154k) | No | Encoder that knows chemistry + masked-atom accuracy |
| B. Fine-tuning | 645 MDM2 compounds (`ro5_properties_filtered.csv`) | Yes (pIC50, Active/Inactive) | pIC50 regressor + activity classifier |
| C. Proof of value | Same 80/20 split, scratch vs pretrained | — | Learning curves + test table: pretrained converges faster & scores higher |

**Run time:** ~5–10 min on Colab T4 with defaults (`PRETRAIN_SUBSET=20000`, 20 + 50 epochs). Set `FULL_RUN=True` for the full 154k × 100-epoch paper run.

In [1]:
# 0. Setup (Colab) — safe to re-run. Locally this cell just prints versions.
import os
IN_COLAB = os.path.isdir('/content')
print('IN_COLAB =', IN_COLAB)
if IN_COLAB:
    # Keep in sync with repo notebooks: torch-geometric + rdkit
    import sys
    get_ipython().system('pip install -q "torch-geometric>=2.5" "rdkit>=2023.09" scikit-learn pandas matplotlib seaborn')
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

IN_COLAB = True
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 60.9 MB/s eta 0:00:00:00:0100:01
torch 2.11.0+cu130 | cuda: True


In [ ]:
# 0b. Data auto-download (GitHub) + local cache — re-runs REUSE files, no re-download
# Files land next to this notebook; later cells find them via their normal search paths.
import os, urllib.request
GH_USER, GH_REPO, GH_BRANCH = 'Techbjd', 'new_method', 'main'
GH_FILES = {  # local name -> path inside the GitHub repo
    'ro5_properties_filtered.csv': 'ro5_properties_filtered.csv',                       # 645 labelled MDM2 compounds
    'screening_results.csv':      'Part_8/screening_results.csv',                        # 154k COCONUT SMILES pool
    'random_forest_model.pkl':    'testing/fix_of_chemphore/random_forest_model.pkl',    # RF consensus model (512-bit Morgan r=3)
}
MIN_BYTES = {'ro5_properties_filtered.csv': 50_000, 'screening_results.csv': 5_000_000,
             'random_forest_model.pkl': 500_000}  # guards against truncated downloads
def _fetch(url, dst, tries=3):
    for t in range(1, tries + 1):
        try:
            urllib.request.urlretrieve(url, dst)
            if os.path.getsize(dst) >= MIN_BYTES.get(os.path.basename(dst), 1):
                return True
            os.remove(dst)
        except Exception as e:
            print(f'  try {t} failed: {e}')
    return False
for local, repo_path in GH_FILES.items():
    if os.path.exists(local) and os.path.getsize(local) >= MIN_BYTES.get(local, 1):
        print(f'REUSE  {local} ({os.path.getsize(local)//1024} KB cached, no download)')
        continue
    url = f'https://raw.githubusercontent.com/{GH_USER}/{GH_REPO}/{GH_BRANCH}/{repo_path}'
    print(f'DOWNLOAD {local} <- {url}')
    ok = _fetch(url, local)
    print(('  saved' if ok else '  FAILED — notebook will fall back (COCONUT->MDM2 SMILES, RF->GNN-only tiers)') + f' {local}')
print('data dir:', os.getcwd(), '| files:', sorted(f for f in GH_FILES if os.path.exists(f)))


In [2]:
# 1. Imports, seed, device, config
import os, random, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torch.nn.functional as F
from rdkit import Chem
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, mean_squared_error, r2_score,
                             matthews_corrcoef)
from sklearn.model_selection import train_test_split
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool, global_max_pool

RANDOM_SEED = 57
def set_seed(s=57):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(RANDOM_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# ---- Config: fast defaults (paper run = FULL_RUN=True) ----
FULL_RUN = False          # True -> full 154k pretrain + 100/100 epochs
PRETRAIN_SUBSET = None if FULL_RUN else 20000   # None = all COCONUT SMILES
PRETRAIN_EPOCHS = 100 if FULL_RUN else 20
FINETUNE_EPOCHS = 100 if FULL_RUN else 50
BATCH_SIZE = 64
MASK_RATIO = 0.15         # fraction of atoms masked per molecule (BERT-style)
HIDDEN, LAYERS, DROPOUT = 128, 3, 0.2
LR, WD = 1e-3, 1e-4
print(f'pretrain_subset={PRETRAIN_SUBSET} | pre_epochs={PRETRAIN_EPOCHS} | ft_epochs={FINETUNE_EPOCHS}')

device: cuda
pretrain_subset=20000 | pre_epochs=20 | ft_epochs=50


## Step 1 — The Masking Task (theory, 60 seconds)

Take a real molecule, e.g. `c1ccncc1` (pyridine). Pick one atom, **hide its element** (replace with a generic `[MASK]` token) but keep everything else: where it sits, how many bonds it has, bond types, aromaticity.

The GNN must output a probability over elements `{C, N, O, F, Cl, S, Br, I}` using **only the neighbour context** passed along bonds:

```
loss = CrossEntropy( predicted_element[masked positions], true_element )
```

No property labels are needed — the molecule *is* the label. That is why we can pretrain on 154k COCONUT natural products for free, then fine-tune on just 600 labelled MDM2 compounds.

**Why it works:** to guess well, the network must internalise valence, aromaticity, heteroatom patterns — exactly the priors a small-data property model lacks.

In [3]:
# 2. Load YOUR labelled data: 645 MDM2 compounds
def _try_read(cands):
    for f in cands:
        try:
            df = pd.read_csv(f)
            print('loaded:', f, df.shape)
            return df, f
        except Exception:
            pass
    return None, None

df, SRC = _try_read(['ro5_properties_filtered.csv',
                     'Part_2/ro5_properties_filtered.csv',
                     'Part_15/ro5_properties_filtered.csv',
                     'Natural_MDM2_Inhibitor_Discovery_using_ML/Part_2/ro5_properties_filtered.csv'])
assert df is not None, 'ro5_properties_filtered.csv not found — put it next to this notebook.'
df = df.dropna(subset=['canonical_smiles']).copy()
df['y_clf'] = (df['bioactivity_class'] == 'Active').astype(int)
print('SOURCE =', SRC, '| n =', len(df))
print(df['bioactivity_class'].value_counts().to_dict(), '| pIC50 mean %.2f' % df['pIC50'].mean())

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(df['pIC50'], bins=30, color='steelblue', edgecolor='white')
ax[0].set_title('pIC50 distribution (n=%d)' % len(df)); ax[0].set_xlabel('pIC50')
sns.countplot(x='bioactivity_class', data=df, ax=ax[1], palette='muted')
ax[1].set_title('Active vs Inactive'); plt.tight_layout(); plt.savefig('mask_label_dist.png', dpi=120); plt.show()

AssertionError: ro5_properties_filtered.csv not found — put it next to this notebook.

In [ ]:
# 3. Load UNLABELLED pretraining pool: COCONUT screening SMILES (no labels used)
coco, CSRC = _try_read(['screening_results.csv',
                        'Part_8/screening_results.csv',
                        'Natural_MDM2_Inhibitor_Discovery_using_ML/Part_8/screening_results.csv'])
if coco is not None:
    smis = coco['canonical_smiles'].dropna().drop_duplicates().tolist()
    print(f'COCONUT pool: {len(smis)} unique SMILES (labels ignored)')
else:
    print('No COCONUT file found — pretraining falls back to MDM2 SMILES (labels still unused).')
    smis = df['canonical_smiles'].tolist()
if PRETRAIN_SUBSET is not None and len(smis) > PRETRAIN_SUBSET:
    rng = np.random.RandomState(RANDOM_SEED)
    smis = list(rng.choice(smis, PRETRAIN_SUBSET, replace=False))
print('pretraining SMILES used:', len(smis))

## Molecules → graphs (what the GNN actually sees)

Atoms = nodes, bonds = edges. Each atom gets an **18-dim feature**: 8-dim element one-hot + 6-dim degree one-hot + formal charge + H-count + aromaticity + `is_mask` flag. When an atom is masked we zero its element one-hot and set `is_mask=1` — structure stays, identity is hidden. Bonds carry a 4-dim type one-hot (kept for GINE upgrades; the GCN encoder below uses connectivity).

In [ ]:
# 4. SMILES -> PyG graph (mask-safe features)
ELEMENTS = ['C', 'N', 'O', 'F', 'Cl', 'S', 'Br', 'I']
ELEM2IDX = {e: i for i, e in enumerate(ELEMENTS)}
N_ELEM = len(ELEMENTS)
FEAT_DIM = 8 + 6 + 1 + 1 + 1 + 1  # = 18
print('elements:', ELEMENTS, '| feat dim:', FEAT_DIM)

def atom_features(atom, masked=False):
    oh = [0.0] * N_ELEM
    if not masked and atom.GetSymbol() in ELEM2IDX:
        oh[ELEM2IDX[atom.GetSymbol()]] = 1.0
    d = min(atom.GetDegree(), 5)
    deg = [1.0 if d == i else 0.0 for i in range(6)]
    return oh + deg + [float(atom.GetFormalCharge()),
                       atom.GetTotalNumHs() / 4.0,
                       1.0 if atom.GetIsAromatic() else 0.0,
                       1.0 if masked else 0.0]

BT = Chem.rdchem.BondType
def bond_features(b):
    t = b.GetBondType()
    return [1.0 if t == x else 0.0 for x in (BT.SINGLE, BT.DOUBLE, BT.TRIPLE, BT.AROMATIC)]

def mol_to_graph(smiles, pic50=None, activity=None):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    xs, elems, eis, eas = [], [], [], []
    for a in mol.GetAtoms():
        xs.append(atom_features(a))
        s = a.GetSymbol()
        elems.append(ELEM2IDX.get(s, -1))  # -1 = outside our 8 (rare Br/I covered; else skipped in loss)
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        eis += [[i, j], [j, i]]
        f = bond_features(b)
        eas += [f, f]
    if not eis:  # single-atom molecule
        eis = [[0, 0]]; eas = [[1.0, 0, 0, 0]]
    g = Data(x=torch.tensor(xs, dtype=torch.float),
             edge_index=torch.tensor(eis, dtype=torch.long).t().contiguous(),
             edge_attr=torch.tensor(eas, dtype=torch.float),
             elem=torch.tensor(elems, dtype=torch.long))
    g.smiles = smiles
    g.y_reg = torch.tensor([float(pic50) if pic50 is not None else 0.0], dtype=torch.float)
    g.y_clf = torch.tensor([int(activity) if activity is not None else 0], dtype=torch.long)
    return g

# sanity check on YOUR data
g0 = mol_to_graph(df['canonical_smiles'].iloc[0], df['pIC50'].iloc[0], df['y_clf'].iloc[0])
print('nodes:', g0.num_nodes, '| edges:', g0.num_edges, '| x:', tuple(g0.x.shape))
from collections import Counter
c = Counter()
for s in df['canonical_smiles']:
    m = Chem.MolFromSmiles(s)
    if m:
        c.update(a.GetSymbol() for a in m.GetAtoms() if a.GetSymbol() in ELEM2IDX)
tot = sum(c.values())
print({k: f'{v} ({v/tot:.1%})' for k, v in c.most_common()})
print('majority baseline (always predict C): %.1f%%' % (100 * c['C'] / tot))

In [ ]:
# 5. The masking operation itself — hide 15% of atoms per molecule
def mask_batch(batch, mask_ratio=MASK_RATIO, rng=None):
    """Dynamically mask nodes. Returns masked_x, mask_idx (global), mask_labels."""
    if rng is None:
        rng = np.random
    b = batch.batch.cpu().numpy()
    idx_all, lab_all = [], []
    for g in np.unique(b):
        loc = np.where(b == g)[0]
        k = max(1, int(round(len(loc) * mask_ratio)))
        sel = rng.choice(loc, size=min(k, len(loc)), replace=False)
        idx_all.append(sel)
        lab_all.append(batch.elem.cpu().numpy()[sel])
    mask_idx = torch.from_numpy(np.concatenate(idx_all)).long().to(batch.x.device)
    mask_labels = torch.from_numpy(np.concatenate(lab_all)).long().to(batch.x.device)
    # keep only labels in our 8-element vocabulary for the loss
    keep = mask_labels >= 0
    masked_x = batch.x.clone()
    masked_x[mask_idx, 0:N_ELEM] = 0.0   # hide element identity
    masked_x[mask_idx, -1] = 1.0         # raise the mask flag
    return masked_x, mask_idx[keep], mask_labels[keep]

# demo: mask one real MDM2 molecule and show what the model sees vs hides
import torch_geometric.utils as _u
_demo = mol_to_graph('Cc1ccncc1', 7.0, 1).to('cpu')
from torch_geometric.loader import DataLoader as _DL
_db = next(iter(_DL([_demo], batch_size=1, shuffle=False)))
_mx, _mi, _ml = mask_batch(_db, mask_ratio=0.2, rng=np.random.RandomState(0))
print('masked node ids:', _mi.tolist(), '| true elements:', [ELEMENTS[i] for i in _ml.tolist()])
print('row before:', _db.x[_mi].tolist())
print('row after :', _mx[_mi].tolist(), '<- element zeroed, is_mask=1')

## Step 2 — The Result: the model learns chemistry rules

If masked-atom accuracy beats the ~74% majority baseline **and** the confusion matrix shows sensible confusions (N↔C in aromatic rings, O↔N near carbonyls, F/Cl on aromatic carbons...), the encoder has absorbed valence and context rules — with zero property labels. The per-element recall cell below is your direct evidence.

In [ ]:
# 6. Build pretraining set (graphs WITHOUT labels) + fine-tune set (graphs WITH labels)
pre_graphs = []
for s in smis:
    g = mol_to_graph(s)
    if g is not None:
        pre_graphs.append(g)
print('pretrain graphs:', len(pre_graphs))

ft_graphs = []
for _, r in df.iterrows():
    g = mol_to_graph(r['canonical_smiles'], r['pIC50'], r['y_clf'])
    if g is not None:
        ft_graphs.append(g)
print('fine-tune graphs:', len(ft_graphs))

n_pre_te = max(512, int(0.05 * len(pre_graphs)))
pre_tr, pre_te = pre_graphs[:-n_pre_te], pre_graphs[-n_pre_te:]
pre_train_loader = DataLoader(pre_tr, batch_size=BATCH_SIZE, shuffle=True)
pre_test_loader = DataLoader(pre_te, batch_size=BATCH_SIZE, shuffle=False)
print(f'pretrain split: {len(pre_tr)} train / {len(pre_te)} held-out')

In [ ]:
# 7. Models: small-data GNN encoder + mask head + property head
class GNNEncoder(nn.Module):
    """3 x GCN (18 -> 128), ~50k params. Small on purpose: n=645 overfits big GNNs."""
    def __init__(self, in_dim=FEAT_DIM, hidden=HIDDEN, layers=LAYERS, dropout=DROPOUT):
        super().__init__()
        self.convs = nn.ModuleList([GCNConv(in_dim if i == 0 else hidden, hidden) for i in range(layers)])
        self.bns = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(layers)])
        self.dropout = dropout
    def forward(self, x, edge_index):
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x).relu()
            x = F.dropout(x, p=self.dropout, training=self.training)
        return x

class MaskedAtomGNN(nn.Module):
    def __init__(self, hidden=HIDDEN, dropout=DROPOUT):
        super().__init__()
        self.encoder = GNNEncoder(hidden=hidden, dropout=dropout)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, N_ELEM))
    def forward(self, x, edge_index):
        return self.head(self.encoder(x, edge_index))

class PropertyGNN(nn.Module):
    """Multi-task: pIC50 regression + Active/Inactive classification."""
    def __init__(self, encoder=None, hidden=HIDDEN, dropout=DROPOUT):
        super().__init__()
        self.encoder = encoder if encoder is not None else GNNEncoder(hidden=hidden, dropout=dropout)
        self.reg = nn.Sequential(nn.Linear(2 * hidden, 64), nn.ReLU(),
                                 nn.Dropout(dropout), nn.Linear(64, 1))
        self.clf = nn.Sequential(nn.Linear(2 * hidden, 64), nn.ReLU(),
                                 nn.Dropout(dropout), nn.Linear(64, 2))
    def forward(self, x, edge_index, batch):
        h = self.encoder(x, edge_index)
        g = torch.cat([global_mean_pool(h, batch), global_max_pool(h, batch)], dim=1)
        return self.reg(g).view(-1), self.clf(g)

_p = MaskedAtomGNN().to(device)
n_params = sum(p.numel() for p in _p.parameters())
print(f'pretrain model params: {n_params:,} | encoder: {sum(p.numel() for p in _p.encoder.parameters()):,}')
assert n_params < 120_000, 'keep it small-data-safe'

## Stage A — Pretrain: guess the hidden atoms (no labels)

In [ ]:
# 8. Pretraining loop (dynamic masking = new masks every epoch, BERT-style)
def eval_masked(model, loader):
    model.eval()
    tot_loss, tot_ok, tot_n = 0.0, 0, 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            mx, mi, ml = mask_batch(batch, rng=np.random)
            if len(mi) == 0:
                continue
            logits = model(mx, batch.edge_index)[mi]
            tot_loss += F.cross_entropy(logits, ml, reduction='sum').item()
            tot_ok += (logits.argmax(1) == ml).sum().item()
            tot_n += len(ml)
    return tot_loss / max(1, tot_n), tot_ok / max(1, tot_n)

mask_model = MaskedAtomGNN().to(device)
opt = torch.optim.Adam(mask_model.parameters(), lr=LR, weight_decay=WD)
pre_tr_loss, pre_te_loss, pre_te_acc = [], [], []
best_acc, best_state = 0.0, None
for ep in range(1, PRETRAIN_EPOCHS + 1):
    mask_model.train()
    run_loss, run_n = 0.0, 0
    rng = np.random.RandomState(RANDOM_SEED + ep)
    for batch in pre_train_loader:
        batch = batch.to(device)
        mx, mi, ml = mask_batch(batch, rng=rng)
        if len(mi) == 0:
            continue
        opt.zero_grad()
        logits = mask_model(mx, batch.edge_index)[mi]
        loss = F.cross_entropy(logits, ml)
        loss.backward(); opt.step()
        run_loss += loss.item() * len(ml); run_n += len(ml)
    tr = run_loss / max(1, run_n)
    te_loss, te_acc = eval_masked(mask_model, pre_test_loader)
    pre_tr_loss.append(tr); pre_te_loss.append(te_loss); pre_te_acc.append(te_acc)
    if te_acc > best_acc:
        best_acc, best_state = te_acc, {k: v.cpu() for k, v in mask_model.state_dict().items()}
    if ep % 5 == 0 or ep == 1:
        print(f'epoch {ep:3d}/{PRETRAIN_EPOCHS}  train_loss {tr:.4f}  heldout_loss {te_loss:.4f}  heldout_acc {te_acc:.3f}')
mask_model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
torch.save(mask_model.state_dict(), 'masked_gnn_pretrain.pth')
print(f'\nBest held-out masked-atom acc: {best_acc:.3f}  (majority-C baseline ~0.74) -> saved masked_gnn_pretrain.pth')

In [ ]:
# 9. Did it learn chemistry? curves + per-element recall (the Step-2 evidence)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(pre_tr_loss, label='train loss'); ax[0].plot(pre_te_loss, label='held-out loss')
ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=0.3); ax[0].set_title('Masked-atom loss')
ax[1].plot(pre_te_acc, 'o-', color='green')
ax[1].axhline(0.74, ls='--', color='red', label='majority-C baseline')
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('accuracy'); ax[1].legend(); ax[1].grid(alpha=0.3)
ax[1].set_title('Held-out masked-atom accuracy'); plt.tight_layout()
plt.savefig('mask_pretrain_curves.png', dpi=120); plt.show()

# per-element recall on held-out pool
mask_model.eval()
ys, ps = [], []
with torch.no_grad():
    for batch in pre_test_loader:
        batch = batch.to(device)
        mx, mi, ml = mask_batch(batch, rng=np.random)
        if len(mi) == 0:
            continue
        pr = mask_model(mx, batch.edge_index)[mi].argmax(1).cpu().tolist()
        ps += pr; ys += ml.cpu().tolist()
cm = confusion_matrix(ys, ps, labels=list(range(N_ELEM)))
rec = cm.diagonal() / cm.sum(1).clip(min=1)
print('per-element recall:', {e: round(float(r), 3) for e, r in zip(ELEMENTS, rec)})
print('overall masked acc: %.3f' % accuracy_score(ys, ps))
plt.figure(figsize=(7, 5.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=ELEMENTS, yticklabels=ELEMENTS)
plt.xlabel('predicted'); plt.ylabel('true'); plt.title('Masked-atom confusion (held-out COCONUT)')
plt.tight_layout(); plt.savefig('mask_confusion.png', dpi=120); plt.show()
print('\nRead it as chemistry: strong C/N/O diagonal = valence + aromaticity learned;')
print('N<->C and O<->N off-diagonal mass = the model exploiting neighbour context, exactly the goal.')

In [ ]:
# 10. Qualitative check: mask EACH atom of a real MDM2 inhibitor, show top-3 guesses
mask_model.eval()
SMI = df['canonical_smiles'].iloc[0]
mol = Chem.MolFromSmiles(SMI)
print('SMILES:', SMI)
g = mol_to_graph(SMI).to(device)
with torch.no_grad():
    for atom_id in range(min(g.num_nodes, 8)):
        xm = g.x.clone()
        xm[atom_id, 0:N_ELEM] = 0.0; xm[atom_id, -1] = 1.0
        prob = mask_model(xm, g.edge_index)[atom_id].softmax(0).cpu()
        top3 = sorted(zip(ELEMENTS, prob.tolist()), key=lambda t: -t[1])[:3]
        nbrs = sorted({g.edge_index[1, e].item() for e in (g.edge_index[0] == atom_id).nonzero().view(-1).tolist()})
        nbr_syms = [mol.GetAtomWithIdx(i).GetSymbol() for i in nbrs]
        true = mol.GetAtomWithIdx(atom_id).GetSymbol()
        print(f'atom {atom_id}: TRUE={true} neighbours={nbr_syms} -> ' + ' / '.join(f'{e} {p:.2f}' for e, p in top3))

## Step 3 — Fine-Tuning on your ~600 compounds

Swap the mask head for a property head, keep the pretrained encoder, train on pIC50 + Active/Inactive (multi-task). The control is the **identical architecture trained from scratch** on the same split — same seeds, same epochs. If pretraining helped, you will see (a) lower loss from epoch 1, (b) steeper curves, (c) better test metrics. That is the "incredibly fast and highly accurate" claim, measured rather than asserted.

In [ ]:
# 11. Same 80/20 stratified split as Parts 9/10/12/13 (seed 57)
idx = np.arange(len(ft_graphs))
y = np.array([int(g.y_clf.item()) for g in ft_graphs])
tr_i, te_i = train_test_split(idx, test_size=0.20, stratify=y, random_state=RANDOM_SEED)
tr_graphs = [ft_graphs[i] for i in tr_i]
te_graphs = [ft_graphs[i] for i in te_i]
print(f'train {len(tr_graphs)} / test {len(te_graphs)}')
print('train active rate %.2f | test active rate %.2f' % (y[tr_i].mean(), y[te_i].mean()))
tr_loader = DataLoader(tr_graphs, batch_size=BATCH_SIZE, shuffle=True)
te_loader = DataLoader(te_graphs, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# 12. Fine-tune: scratch (random init) vs pretrained encoder — same protocol
def make_finetune(use_pretrained, dropout=DROPOUT):
    enc = GNNEncoder(dropout=dropout)
    if use_pretrained:
        sd = torch.load('masked_gnn_pretrain.pth', map_location='cpu')
        enc.load_state_dict({k.replace('encoder.', ''): v for k, v in sd.items() if k.startswith('encoder.')})
    return PropertyGNN(encoder=enc, dropout=dropout).to(device)

def run_finetune(model, loader_tr, loader_te, epochs=FINETUNE_EPOCHS, lr=LR, tag=''):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=WD)
    ce, mse = nn.CrossEntropyLoss(), nn.MSELoss()
    hist = {'loss': [], 'auc': [], 'r2': []}
    best = (-1, None)
    for ep in range(1, epochs + 1):
        model.train()
        for b in loader_tr:
            b = b.to(device)
            pr, pc = model(b.x, b.edge_index, b.batch)
            loss = mse(pr, b.y_reg.view(-1)) + ce(pc, b.y_clf.view(-1))
            opt.zero_grad(); loss.backward(); opt.step()
        # monitor on test (same convention as Parts 9/10)
        model.eval()
        P, T, PR, TR = [], [], [], []
        with torch.no_grad():
            for b in loader_te:
                b = b.to(device)
                pr, pc = model(b.x, b.edge_index, b.batch)
                P += pc.softmax(1)[:, 1].cpu().tolist()
                T += b.y_clf.view(-1).cpu().tolist()
                PR += pr.cpu().tolist(); TR += b.y_reg.view(-1).cpu().tolist()
        try:
            auc = roc_auc_score(T, P)
        except ValueError:
            auc = float('nan')
        r2 = r2_score(TR, PR)
        hist['loss'].append(mse(torch.tensor(PR), torch.tensor(TR)).item())
        hist['auc'].append(auc); hist['r2'].append(r2)
        key = (auc if not np.isnan(auc) else -1) + max(r2, -1)
        if key > best[0]:
            best = (key, {k: v.cpu() for k, v in model.state_dict().items()})
    model.load_state_dict({k: v.to(device) for k, v in best[1].items()})
    return hist

scratch = make_finetune(False)
pretrained = make_finetune(True)
h_scratch = run_finetune(scratch, tr_loader, te_loader, tag='scratch')
h_pre = run_finetune(pretrained, tr_loader, te_loader, tag='pretrained')
torch.save(scratch.state_dict(), 'property_gnn_scratch.pth')
torch.save(pretrained.state_dict(), 'property_gnn_finetuned.pth')
print('saved property_gnn_scratch.pth + property_gnn_finetuned.pth')

In [ ]:
# 13. Learning-speed plot: the core Step-3 evidence
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(h_scratch['auc'], label='scratch'); ax[0].plot(h_pre['auc'], label='pretrained')
ax[0].set_title('Test ROC-AUC vs epoch'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(h_scratch['r2'], label='scratch'); ax[1].plot(h_pre['r2'], label='pretrained')
ax[1].set_title('Test R2 (pIC50) vs epoch'); ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig('finetune_speed.png', dpi=120); plt.show()
print('Expectation: pretrained curve starts higher and plateaus earlier (fewer epochs to same score).')
print('If curves tie, increase PRETRAIN_SUBSET / PRETRAIN_EPOCHS — 645 labels are too few for scratch to catch a good prior.')

In [ ]:
# 14. Test-set scoreboard: scratch vs fine-tuned (classification + regression)
def evaluate_prop(model, loader):
    model.eval()
    P, T, PR, TR = [], [], [], []
    with torch.no_grad():
        for b in loader:
            b = b.to(device)
            pr, pc = model(b.x, b.edge_index, b.batch)
            P += pc.argmax(1).cpu().tolist()
            T += b.y_clf.view(-1).cpu().tolist()
            PR += pr.cpu().tolist(); TR += b.y_reg.view(-1).cpu().tolist()
    P, T = np.array(P), np.array(T)
    PR, TR = np.array(PR), np.array(TR)
    proba = []
    with torch.no_grad():
        for b in loader:
            b = b.to(device)
            _, pc = model(b.x, b.edge_index, b.batch)
            proba += pc.softmax(1)[:, 1].cpu().tolist()
    return {'acc': accuracy_score(T, P), 'prec': precision_score(T, P, zero_division=0),
            'rec': recall_score(T, P, zero_division=0), 'f1': f1_score(T, P, zero_division=0),
            'auc': roc_auc_score(T, proba), 'mcc': matthews_corrcoef(T, P),
            'rmse': mean_squared_error(TR, PR) ** 0.5, 'r2': r2_score(TR, PR),
            'P': proba, 'T': T, 'PR': PR, 'TR': TR}

res_s = evaluate_prop(scratch, te_loader)
res_p = evaluate_prop(pretrained, te_loader)
board = pd.DataFrame([{'model': 'scratch', **{k: v for k, v in res_s.items() if k not in 'PTPRTR'}},
                      {'model': 'pretrained-finetuned', **{k: v for k, v in res_p.items() if k not in 'PTPRTR'}}])
print(board[['model', 'acc', 'prec', 'f1', 'rec', 'auc', 'mcc', 'rmse', 'r2']].round(3).to_string(index=False))
board.to_csv('performance_mask_vs_scratch.csv', index=False)
print('\nsaved performance_mask_vs_scratch.csv')
print('\nContext (same 80/20 family, seed 57): RF 0.960 / GCN-scratch 0.952 / Hybrid 0.956 AUC.')
print('This notebook trades a little peak AUC for sample-efficiency + speed on n=645.')

In [ ]:
# 15. ROC + confusion + parity plots (fine-tuned model)
from sklearn.metrics import RocCurveDisplay
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
RocCurveDisplay.from_predictions(res_p['T'], res_p['P'], ax=ax[0])
ax[0].set_title('ROC (fine-tuned) AUC=%.3f' % res_p['auc'])
cm = confusion_matrix(res_p['T'], (np.array(res_p['P']) > 0.5).astype(int))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=ax[1])
ax[1].set_title('Confusion (fine-tuned)'); ax[1].set_xlabel('pred'); ax[1].set_ylabel('true')
ax[2].scatter(res_p['TR'], res_p['PR'], alpha=0.6)
ax[2].plot([3, 11], [3, 11], 'r--')
ax[2].set_xlabel('true pIC50'); ax[2].set_ylabel('pred pIC50')
ax[2].set_title('Parity (fine-tuned) R2=%.3f' % res_p['r2'])
plt.tight_layout(); plt.savefig('finetune_eval.png', dpi=120); plt.show()

In [ ]:
# 16a. CORRECTNESS 1/4 — Calibration: is P(active)=0.9 right ~90% of the time?
# A model can have good AUC yet be dangerously overconfident. ECE (lower = better) decides
# whether you may trust probabilities for T1/T2 tiering.
from sklearn.calibration import calibration_curve
def ece_score(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true, dtype=float); y_prob = np.asarray(y_prob, dtype=float)
    edges = np.linspace(0.0, 1.0, n_bins + 1); ece = 0.0
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        m = ((y_prob > lo) & (y_prob <= hi)) if i > 0 else ((y_prob >= lo) & (y_prob <= hi))
        if m.sum() == 0:
            continue
        ece += (m.sum() / len(y_prob)) * abs(float(y_prob[m].mean()) - float(y_true[m].mean()))
    return ece

for name, r in [('scratch', res_s), ('pretrained-finetuned', res_p)]:
    print(f'{name:22s} ECE = {ece_score(r["T"], r["P"]):.3f}  (lower = more trustworthy)')
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for a, (name, r) in zip(ax, [('scratch', res_s), ('pretrained-finetuned', res_p)]):
    frac, mean = calibration_curve(r['T'], r['P'], n_bins=10, strategy='uniform')
    a.plot(mean, frac, 'o-', label=name); a.plot([0, 1], [0, 1], 'k--', label='perfect')
    a.set_xlabel('mean predicted P(active)'); a.set_ylabel('fraction truly active')
    a.set_title(f'{name} (ECE={ece_score(r["T"], r["P"]):.3f})'); a.legend(); a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('finetune_calibration.png', dpi=120); plt.show()
print('Rule: tier/dock on the model with LOWER ECE when AUCs tie.')

In [ ]:
# 16b. CORRECTNESS 2/4 — Precision@k: of the top-k predicted actives, how many are real?
# This is the finding metric: your hit list IS a top-k list. k values fit n_test=129.
def precision_at_k(y_true, y_score, ks=(10, 20, 50)):
    order = np.argsort(-np.asarray(y_score, dtype=float))
    yt = np.asarray(y_true)[order]
    return {k: float(yt[:min(k, len(yt))].mean()) for k in ks}
KS = (10, 20, 50)
pk = pd.DataFrame([{'model': 'scratch', **{f'P@{k}': v for k, v in precision_at_k(res_s['T'], res_s['P'], KS).items()}},
                   {'model': 'pretrained-finetuned', **{f'P@{k}': v for k, v in precision_at_k(res_p['T'], res_p['P'], KS).items()}}])
print(pk.to_string(index=False))
pk.to_csv('precision_at_k.csv', index=False)
print('\nsaved precision_at_k.csv — report this next to AUC; it answers "are my top hits correct?"')

In [ ]:
# 16c. CORRECTNESS 3/4 — Applicability domain: max Morgan Tanimoto to TRAIN (cut 0.35, locked)
from rdkit.Chem import AllChem, DataStructs
AD_CUT = 0.35
def morgan_fp(smi, r=2, n=2048):
    m = Chem.MolFromSmiles(smi)
    return AllChem.GetMorganFingerprintAsBitVect(m, r, nBits=n) if m is not None else None
tr_smis = [ft_graphs[i].smiles for i in tr_i]
te_smis = [ft_graphs[i].smiles for i in te_i]
tr_fps = [f for f in (morgan_fp(s) for s in tr_smis) if f is not None]
print(f'train fingerprints: {len(tr_fps)}/{len(tr_smis)}')
def max_tanimoto(smi):
    f = morgan_fp(smi)
    if f is None or not tr_fps:
        return 0.0
    return max(DataStructs.TanimotoSimilarity(f, t) for t in tr_fps)
te_ad = np.array([max_tanimoto(s) for s in te_smis])
te_in_ad = te_ad >= AD_CUT
print(f'in-AD test rate: {te_in_ad.mean():.2f} ({te_in_ad.sum()}/{len(te_in_ad)}) at cut {AD_CUT}')
for tag, m in [('in-AD', te_in_ad), ('OOD', ~te_in_ad)]:
    if m.sum() == 0:
        print(f'{tag}: n=0'); continue
    T = np.array(res_p['T'])[m]; P = (np.array(res_p['P'])[m] > 0.5).astype(int)
    print(f"{tag} (n={m.sum()}): acc={accuracy_score(T, P):.3f} prec={precision_score(T, P, zero_division=0):.3f}")
pd.DataFrame({'smiles': te_smis, 'true': list(np.array(res_p['T'])),
              'pred_pIC50': list(res_p['PR']), 'P_active': list(res_p['P']),
              'max_tanimoto': te_ad, 'in_ad': te_in_ad}).to_csv('ad_assessment.csv', index=False)
print('saved ad_assessment.csv')

In [ ]:
# 16d. CORRECTNESS 4/4 — RF consensus + T1/T2/T3 tiers (ensemble stays PRIMARY ranker)
# Locked rule: T1 = GNN pIC50>=7.5 & RF>0.6 & in-AD (dock first); T2 = GNN-only (novelty appendix); else T3.
import joblib
RF_CANDS = ['random_forest_model.pkl', 'Part_6/random_forest_model.pkl',
            'Natural_MDM2_Inhibitor_Discovery_using_ML/random_forest_model.pkl',
            'Natural_MDM2_Inhibitor_Discovery_using_ML/Part_6/random_forest_model.pkl']
rf, RF_SRC = None, None
for f in RF_CANDS:
    try:
        rf = joblib.load(f); RF_SRC = f; break
    except Exception:
        pass
print('RF model:', RF_SRC if rf is not None else 'NOT FOUND -> tiers use GNN-only (T1 unreachable, by design)')
def morgan512(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return None
    bv = AllChem.GetMorganFingerprintAsBitVect(m, 3, nBits=512)
    arr = np.zeros((512,), dtype=float); DataStructs.ConvertToNumpyArray(bv, arr)
    return arr
if rf is not None:
    Xte = np.stack([morgan512(s) for s in te_smis])
    rf_p = rf.predict_proba(Xte)[:, 1]
else:
    rf_p = np.full(len(te_smis), np.nan)
gnn_pic = np.array(res_p['PR']); gnn_p = np.array(res_p['P']); yte = np.array(res_p['T'])
tiers = []
for i in range(len(te_smis)):
    g_hit = bool(gnn_pic[i] >= 7.5)
    r_hit = bool(rf is not None and rf_p[i] > 0.6)
    ad = bool(te_in_ad[i])
    tiers.append('T1' if (g_hit and r_hit and ad) else ('T2' if g_hit else 'T3'))
tiers = np.array(tiers)
for t in ('T1', 'T2', 'T3'):
    m = tiers == t
    print(f'{t}: n={m.sum()}' + (f' | precision={yte[m].mean():.3f} (true-active rate in tier)' if m.sum() else ''))
pd.DataFrame({'smiles': te_smis, 'true': yte, 'gnn_pIC50': gnn_pic, 'gnn_P': gnn_p,
              'rf_P': rf_p, 'max_tanimoto': te_ad, 'in_ad': te_in_ad, 'tier': tiers}
             ).to_csv('consensus_tiers.csv', index=False)
print('saved consensus_tiers.csv — T1 rows are your dock-first list.')

In [ ]:
# 16. Honesty check: Y-randomization (3 shuffles, short runs) — must collapse to ~random
ce_, mse_ = nn.CrossEntropyLoss(), nn.MSELoss()
for s in range(3):
    rng = np.random.RandomState(1000 + s)
    shuf = [ft_graphs[i] for i in tr_i]
    # shuffle labels across train graphs
    prm = rng.permutation(len(shuf))
    regs = [float(ft_graphs[i].y_reg.item()) for i in tr_i]
    clfs = [int(ft_graphs[i].y_clf.item()) for i in tr_i]
    for g, j in zip(shuf, prm):
        g = g  # keep graph, assign shuffled label copy
    import copy
    shuf_c = []
    for g, j in zip(shuf, prm):
        gc = copy.copy(g)
        gc.y_reg = torch.tensor([regs[j]], dtype=torch.float)
        gc.y_clf = torch.tensor([clfs[j]], dtype=torch.long)
        shuf_c.append(gc)
    m = make_finetune(True).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=LR, weight_decay=WD)
    sl = DataLoader(shuf_c, batch_size=BATCH_SIZE, shuffle=True)
    for _ in range(10):
        m.train()
        for b in sl:
            b = b.to(device)
            pr, pc = m(b.x, b.edge_index, b.batch)
            loss = mse_(pr, b.y_reg.view(-1)) + ce_(pc, b.y_clf.view(-1))
            opt.zero_grad(); loss.backward(); opt.step()
    r = evaluate_prop(m, te_loader)
    print(f'shuffle {s}: AUC {r["auc"]:.3f} | R2 {r["r2"]:.3f}  (expect ~0.5 / <=0)')
print('If shuffled runs score like the real run, the pipeline leaks — it should not.')

## Stage B — Screen COCONUT: GNN + RF + AD + medchem -> tiered hit list
Runs on the fast subset by default (`FULL_RUN=False` -> 20k). Set `FULL_RUN=True` for all ~154k (~1-2 h).
Every hit carries both votes + AD + medchem flags, so a "finding" is never a bare ML score.

In [ ]:
# 18. COCONUT screen (Colab-ready): predict -> RF-consensus -> AD -> PAINS/Brenk -> tiers
import time
t0 = time.time()
SCREEN_N = None if FULL_RUN else 20000
if coco is None:
    print('SKIP: no COCONUT table found — tiers already proven on the test set (consensus_tiers.csv).')
    screen_df = None
else:
    pool = coco['canonical_smiles'].dropna().drop_duplicates().tolist()
    print(f'COCONUT pool: {len(pool)}')
    if SCREEN_N is not None and len(pool) > SCREEN_N:
        pool = list(np.random.RandomState(RANDOM_SEED).choice(pool, SCREEN_N, replace=False))
    print(f'screening n={len(pool)}')
    # --- GNN forward ---
    sg, ss = [], []
    for s in pool:
        g = mol_to_graph(s)
        if g is not None:
            sg.append(g); ss.append(s)
    print(f'featurized {len(sg)}/{len(pool)}')
    sl = DataLoader(sg, batch_size=256, shuffle=False)
    pretrained.eval()
    gp_pic, gp_p = [], []
    with torch.no_grad():
        for b in sl:
            b = b.to(device)
            pr, pc = pretrained(b.x, b.edge_index, b.batch)
            gp_pic += pr.cpu().tolist(); gp_p += pc.softmax(1)[:, 1].cpu().tolist()
    out = pd.DataFrame({'smiles': ss, 'gnn_pIC50': gp_pic, 'gnn_P': gp_p})
    # --- RF consensus (vectorized) ---
    if rf is not None:
        X = np.stack([morgan512(s) for s in ss])
        out['rf_P'] = rf.predict_proba(X)[:, 1]
    else:
        out['rf_P'] = np.nan
    # --- AD ---
    out['max_tanimoto'] = [max_tanimoto(s) for s in ss]
    out['in_ad'] = out['max_tanimoto'] >= AD_CUT
    # --- medchem: PAINS + Brenk (RDKit FilterCatalog) ---
    from rdkit.Chem import FilterCatalog
    _pp = FilterCatalog.FilterCatalogParams()
    _pp.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
    PAINS = FilterCatalog.FilterCatalog(_pp)
    _bp = FilterCatalog.FilterCatalogParams()
    _bp.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
    BRENK = FilterCatalog.FilterCatalog(_bp)
    def medchem_row(smi):
        m = Chem.MolFromSmiles(smi)
        if m is None:
            return False, False
        return (not PAINS.HasMatch(m), not BRENK.HasMatch(m))
    mc = [medchem_row(s) for s in ss]
    out['pains_pass'] = [a for a, _ in mc]; out['brenk_pass'] = [b for _, b in mc]
    out['medchem_pass'] = out['pains_pass'] & out['brenk_pass']
    # --- tiers (locked rule) ---
    def tier_row(r):
        g_hit = r['gnn_pIC50'] >= 7.5
        r_hit = pd.notna(r['rf_P']) and r['rf_P'] > 0.6
        if g_hit and r_hit and r['in_ad'] and r['medchem_pass']:
            return 'T1'
        return 'T2' if g_hit else 'T3'
    out['tier'] = out.apply(tier_row, axis=1)
    print(out['tier'].value_counts().to_dict())
    out.to_csv('coconut_mask_screen_full.csv', index=False)
    t1 = out[out['tier'] == 'T1'].sort_values('gnn_pIC50', ascending=False)
    t1.to_csv('coconut_mask_hits_T1.csv', index=False)
    print(f'dock-first T1 hits: {len(t1)} -> coconut_mask_hits_T1.csv (full table: coconut_mask_screen_full.csv)')
    print(f'screen time: {(time.time()-t0)/60:.1f} min')
    screen_df = out

## Use it: predict new compounds + what to run next

- **Your 600 compounds** → `ro5_properties_filtered.csv` with columns `canonical_smiles, pIC50, bioactivity_class`. Nothing else to change.
- **Colab run sheet:** T4 GPU → upload this notebook + the two CSVs → Run all (~5–10 min defaults; ~1–2 h for `FULL_RUN=True`).
- **Outputs to rescue:** `masked_gnn_pretrain.pth`, `property_gnn_finetuned.pth`, `performance_mask_vs_scratch.csv`, `precision_at_k.csv`, `consensus_tiers.csv`, `ad_assessment.csv`, `coconut_mask_hits_T1.csv`, `coconut_mask_screen_full.csv`, `mask_*.png`, `finetune_*.png`.
- **Next:** dock `coconut_mask_hits_T1.csv` into 4HG7 (T1 = dock first; T2 = novelty appendix), then ADMET.

In [ ]:
# 17. Inference demo: 5 SMILES -> pIC50 + P(active) with the FINE-TUNED model
pretrained.eval()
queries = ['Cc1ccncc1', 'CC(=O)Nc1ccc(O)cc1', 'C1=CC=C(C=C1)C(=O)O',
           df['canonical_smiles'].iloc[0], 'CN1C=NC2=C1C(=O)N(C)C(=O)N2C']
qs = [mol_to_graph(s) for s in queries]
qs = [g for g in qs if g is not None]
ql = DataLoader(qs, batch_size=len(qs), shuffle=False)
with torch.no_grad():
    for b in ql:
        b = b.to(device)
        pr, pc = pretrained(b.x, b.edge_index, b.batch)
        pa = pc.softmax(1)[:, 1].cpu().tolist()
        for s, v, p in zip(queries, pr.cpu().tolist(), pa):
            print(f'pIC50 {v:5.2f} | P(active) {p:.2f} | {s[:60]}')
print('\nFiles written:', [f for f in sorted(os.listdir(".")) if f.startswith(("mask", "finetune", "performance_mask", "property_gnn", "masked_gnn"))])
print('=== Done ===  Pretrain (mask) -> fine-tune (your 600) -> scratch-vs-pretrained proof.')